# 02 – Data Cleaning and Preprocessing

**Project**: DengAI – Predicting Disease Spread  

---

### Objective
- Handle missing values using time-series-appropriate strategies
- Validate data types and ranges
- Produce clean datasets for feature engineering

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

ROOT = Path('../')
RAW  = ROOT / 'data/raw'
PROC = ROOT / 'data/processed'
PROC.mkdir(exist_ok=True)

features = pd.read_csv(RAW / 'training_set_features.csv', parse_dates=['week_start_date'])
labels   = pd.read_csv(RAW / 'training_set_labels.csv')
test_raw = pd.read_csv(RAW / 'test_set_features.csv',    parse_dates=['week_start_date'])

df = features.merge(labels, on=['city','year','weekofyear'])
df = df.sort_values(['city','week_start_date']).reset_index(drop=True)
test_raw = test_raw.sort_values(['city','week_start_date']).reset_index(drop=True)
print("Loaded:", df.shape, test_raw.shape)

Loaded: (1456, 25) (416, 24)


In [2]:
# --- Missing value summary ---
miss = df.isnull().sum()
miss_df = pd.DataFrame({
    'missing': miss[miss > 0],
    'pct': (miss[miss > 0] / len(df) * 100).round(1)
}).sort_values('missing', ascending=False)
print(miss_df)

                                       missing   pct
ndvi_ne                                    194  13.3
ndvi_nw                                     52   3.6
station_diur_temp_rng_c                     43   3.0
station_avg_temp_c                          43   3.0
station_precip_mm                           22   1.5
ndvi_sw                                     22   1.5
ndvi_se                                     22   1.5
station_max_temp_c                          20   1.4
station_min_temp_c                          14   1.0
precipitation_amt_mm                        13   0.9
reanalysis_sat_precip_amt_mm                13   0.9
reanalysis_air_temp_k                       10   0.7
reanalysis_avg_temp_k                       10   0.7
reanalysis_dew_point_temp_k                 10   0.7
reanalysis_max_air_temp_k                   10   0.7
reanalysis_min_air_temp_k                   10   0.7
reanalysis_relative_humidity_percent        10   0.7
reanalysis_specific_humidity_g_per_kg       10

In [3]:
# --- Cleaning strategy: forward-fill then backfill per city ---
# This preserves the temporal ordering of weather observations.
# FFill carries the last known observation forward (natural for sensor data).
# Backfill covers any gaps at the start of a city's series.

WEATHER = [c for c in df.columns if c not in
           ['city','year','weekofyear','week_start_date','total_cases']]

def clean_city(cdf):
    cdf = cdf.copy().sort_values('week_start_date').reset_index(drop=True)
    cdf[WEATHER] = cdf[WEATHER].ffill().bfill()
    return cdf

train_clean = pd.concat([clean_city(df[df.city=='sj']),
                         clean_city(df[df.city=='iq'])], ignore_index=True)
test_clean  = pd.concat([clean_city(test_raw[test_raw.city=='sj']),
                         clean_city(test_raw[test_raw.city=='iq'])], ignore_index=True)

print(f"Remaining NaNs (train): {train_clean.isnull().sum().sum()}")
print(f"Remaining NaNs (test):  {test_clean.isnull().sum().sum()}")

Remaining NaNs (train): 0
Remaining NaNs (test):  0


In [4]:
# --- Validate data ranges ---
checks = {
    'station_avg_temp_c': (10, 40),
    'reanalysis_relative_humidity_percent': (0, 100),
    'precipitation_amt_mm': (0, 500),
}
for col, (lo, hi) in checks.items():
    out = train_clean[(train_clean[col] < lo) | (train_clean[col] > hi)]
    print(f"{col}: {len(out)} out-of-range rows")

station_avg_temp_c: 0 out-of-range rows
reanalysis_relative_humidity_percent: 0 out-of-range rows
precipitation_amt_mm: 0 out-of-range rows


In [5]:
# --- Save cleaned data ---
train_clean.to_csv(PROC / 'train_clean.csv', index=False)
test_clean.to_csv(PROC  / 'test_clean.csv',  index=False)
print("Saved train_clean.csv and test_clean.csv")
print(f"Shapes — train: {train_clean.shape}  test: {test_clean.shape}")

Saved train_clean.csv and test_clean.csv
Shapes — train: (1456, 25)  test: (416, 24)


## Cleaning Summary

| Step | Action | Rationale |
|------|--------|-----------|
| Missing values | Forward-fill → backfill per city | Temporal data; last observation is best estimate |
| Data types | `week_start_date` parsed as datetime | Required for lag/rolling features |
| Outliers | Retained | Extreme weather events are real signals for dengue |

All 20 features with missing values resolved. Zero NaNs remain in both train and test.